PostgreSQL uses a **process-based architecture** rather than a thread-based architecture. Instead of running a single process with multiple threads, PostgreSQL spawns distinct OS-level background and worker processes that share a central memory region called **Shared Memory**.

---

### Process-Based vs. Thread-Based Architecture

| Feature | PostgreSQL (Process-Based) | Thread-Based DBs (e.g., MySQL, SQL Server) |
| --- | --- | --- |
| **Execution Model** | Separate OS processes per connection and background task | Single/few OS processes running multiple threads |
| **Memory Isolation** | High — isolated virtual address space per process | Low — shared memory space across threads |
| **Fault Tolerance** | A crash in one backend process rarely brings down the whole engine | An unhandled thread crash can crash the entire database process |
| **IPC Overhead** | High — relies on OS Shared Memory (System V / POSIX IPC) and semaphores | Low — direct shared variables within process memory space |

---

### Core Memory Structure: Shared Buffer Pool

Because PostgreSQL uses independent processes, data sharing and caching happen through **Shared Memory**:

* **Shared Buffer Pool:** Caches table and index pages read from disk so subsequent reads happen in RAM.
* **Buffer Management:** When a client queries data, its dedicated backend process searches the Shared Buffer Pool first. If missing (a buffer miss), the process reads the block from disk into the buffer pool.
* **Dirty Pages:** When data is modified (INSERT/UPDATE/DELETE), the backend updates the page in the Shared Buffer Pool and marks it as "dirty."

---

### Key PostgreSQL Processes

#### 1. Backend Processes (Client Connections)

* **Role:** Handles individual client connections.
* **Behavior:** When a client connects, the master process (`postmaster`) forks a dedicated backend process for that connection.
* **Function:** Parses, optimizes, and executes SQL queries. It directly interacts with the Shared Buffer Pool to read/write pages and writes transaction logs to the WAL buffers.

#### 2. WAL Writer (Write-Ahead Logging)

* **Role:** Flushes transaction logs from RAM to physical disk.
* **Behavior:** PostgreSQL follows a strict Write-Ahead Logging protocol: changes to data pages **must** be written to disk in the WAL file *before* the dirty data page itself is written to disk.
* **Function:** Periodically writes and flushes WAL buffers from Shared Memory to disk to ensure Durability (the "D" in ACID), allowing crash recovery if power fails unexpectedly.

#### 3. Checkpointer

* **Role:** Flushes dirty data pages from the Shared Buffer Pool to disk.
* **Behavior:** Runs periodically or when the WAL reaches a defined threshold (`checkpoint_completion_target`).
* **Function:** Writes all dirty pages cached in memory back to actual table and index files on disk. Creating this restore point ensures that during recovery, PostgreSQL only needs to replay WAL logs recorded *after* the latest checkpoint.

---

### How They Interact: A Write Operation Example

1. **Client Request:** The client sends an `UPDATE` query to its assigned **Backend Process**.
2. **Buffer Update:** The backend locates the page in the **Shared Buffer Pool**, modifies it, and marks the page as **dirty**.
3. **WAL Record:** The backend writes a log entry describing the change into the Shared Memory WAL buffers.
4. **Transaction Commit:** When the user commits:
* The **WAL Writer** flushes the WAL buffer to disk (ensuring crash safety).
* The backend reports success to the client immediately — the actual table page on disk does *not* need to be updated yet.


5. **Checkpointing:** Later, the **Checkpointer** process asynchronously writes the dirty pages from the **Shared Buffer Pool** out to the main table files on disk.

Connection pooling solves a fundamental performance bottleneck in process-based database architectures like PostgreSQL: **creating a new database connection for every incoming request is extremely expensive**.

In PostgreSQL, establishing a connection requires:

1. An OS-level TCP handshake and TLS negotiation.
2. The `postmaster` process executing a `fork()` system call to spawn a brand-new OS backend process.
3. Allocating process-specific memory space (such as `work_mem` and execution stack).
4. Authenticating the user and reading startup settings.

Connection pooling replaces this per-request overhead with a **reusable pool of warm, pre-established database processes**.

---

### How Connection Pooling Works Under the Hood

A connection pooler (like **PgBouncer**, **pg_cat**, or application-level poolers like **HikariCP**) sits as an intermediary proxy between your application instances and the database server.

```
+------------------+         +----------------------+         +---------------------+
| Application      |         | Connection Pooler    |         | PostgreSQL Engine   |
| (1000s of threads|         | (e.g., PgBouncer)    |         | (Fixed Backend Pool)|
+------------------+         +----------------------+         +---------------------+
| Thread 1 -------|--------> | App Pool (Frontends) |         |                     |
| Thread 2 -------|--------> |                      | ======> | Server Backend 1    |
| Thread 3 -------|--------> | Active Mapping Layer | ======> | Server Backend 2    |
| Thread 4 -------|--------> |                      | ======> | Server Backend 3    |
| Thread N -------|--------> | Queue / Waiting Pool |         |                     |
+------------------+         +----------------------+         +---------------------+

```

#### 1. Startup & Warm Connections

When the connection pooler initializes, it opens a fixed number of persistent connections (e.g., 20 to 50) directly to the PostgreSQL engine. These PostgreSQL backend processes remain alive and idle on the database server, bypassing `fork()` costs during runtime.

#### 2. Request Handshake & Interception

* When an application thread needs to execute a query, it requests a connection from the pooler using standard database driver protocols.
* Instead of talking directly to PostgreSQL, the application connects to the pooler's lightweight socket.
* The pooler evaluates its available server backends:
* **If a server backend is free:** The pooler maps the application request directly to that backend.
* **If all server backends are busy:** The pooler puts the application request into a client queue (up to a configurable timeout limit).



#### 3. Execution & Multiplexing

The pooler routes incoming SQL commands over the pre-established server connection, receives the query results from PostgreSQL, and streams them back to the client application.

#### 4. Connection Reset & Release

When the application finishes its unit of work and calls `connection.close()`, the application-to-pooler socket is closed or returned to the client pool, but **the underlying PostgreSQL backend process is NOT closed**.

Before making that PostgreSQL backend process available to another client thread, the pooler ensures session hygiene (e.g., issuing `DISCARD ALL` or `RESET ALL` in session mode) to wipe temporary tables, prepared statements, and session parameters.

---

### Common Pooling Modes

Connection poolers operate in different modes depending on when a connection is returned to the pool:

| Pooling Mode | Connection Bound & Returned When... | Best Used For | Trade-offs |
| --- | --- | --- | --- |
| **Session Pooling** | A client logs in and remains assigned until explicit disconnect. | Legacy apps, heavy use of `SET` variables or temporary tables. | Lowest concurrency gains; limit matches the database backend limit. |
| **Transaction Pooling** | Assigned when a `BEGIN` statement starts and released on `COMMIT` / `ROLLBACK`. | Microservices, web apps, REST APIs (most popular for Postgres). | Disables session-level state (`LISTEN/NOTIFY`, `SET LOCAL`, temp tables). |
| **Statement Pooling** | Assigned for a single SQL statement and immediately released. | Simple multi-statement auto-commit read queries. | Multi-statement transactions (`BEGIN ... COMMIT`) cannot be used. |

---

### Why Connection Pooling Is Critical for PostgreSQL

Because PostgreSQL assigns a dedicated OS process to each client connection, running hundreds or thousands of direct connections leads to:

* **High Memory Overhead:** Each process consumes megabytes of RAM even when idle.
* **CPU Context-Switch Thrashing:** The OS kernel wastes CPU cycles swapping context between thousands of active database processes.
* **Cache Eviction:** Large numbers of active processes degrade CPU cache efficiency.

By introducing a connection pooler, you can serve **10,000+ incoming web client connections** using a tightly controlled, high-throughput pool of just **20-50 PostgreSQL backend processes**, keeping CPU context switching to a minimum while maximizing throughput.